# Clasificador de productos de inventario

Entrenamiento de un modelo de clasificación de imágenes en **6 categorías** que
corresponden a productos reales del catálogo: *laptop, monitor, teclado, mouse,
silla e impresora*.

El modelo se integra con el sistema de inventario desplegado en AWS: la interfaz
web permite subir una fotografía y el servicio devuelve la categoría predicha.

**Estrategia:** transfer learning sobre MobileNetV3-Large preentrenada en
ImageNet. Se eligió esta arquitectura por su relación entre precisión y tamaño
(≈5,4 M de parámetros), ya que el modelo debe ejecutarse en CPU dentro de un
contenedor con memoria acotada.

**Salida:** un archivo ONNX que consume el servicio de inferencia.

## 1. Configuración

In [1]:
import json, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models

SEMILLA = 7
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

# El M4 Pro expone su GPU a PyTorch a través de Metal Performance Shaders.
dispositivo = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", dispositivo)

PyTorch: 2.14.0
Dispositivo: mps


In [2]:
# Localiza la raíz del proyecto subiendo hasta encontrar datos/dataset
BASE = Path.cwd()
while not (BASE / "datos" / "dataset").exists() and BASE != BASE.parent:
    BASE = BASE.parent

DATOS = BASE / "datos" / "dataset"
SALIDA = BASE / "inventory-ml" / "modelo"
SALIDA.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATOS)
print("Modelo :", SALIDA)

Dataset: /Users/rafaellondonobotero/Documents/Especialización/Computación en la nube/Actividad 3/datos/dataset
Modelo : /Users/rafaellondonobotero/Documents/Especialización/Computación en la nube/Actividad 3/inventory-ml/modelo


## 2. Carga de datos

Las imágenes provienen de Open Images V7, recortadas por su caja delimitadora
(ver `scripts/preparar_dataset.py`). El recorte es necesario porque Open Images
etiqueta escenas completas: una fotografía de escritorio contiene laptop,
teclado y mouse simultáneamente, y entrenar con la imagen entera enseñaría
asociaciones falsas entre clases.

El aumento de datos se aplica solo a entrenamiento. Validación y prueba usan
una transformación determinista para que las métricas sean comparables entre
épocas.

In [3]:
MEDIA = [0.485, 0.456, 0.406]   # estadísticas de ImageNet
DESV  = [0.229, 0.224, 0.225]   # requeridas por los pesos preentrenados
TAM = 224

aumento = transforms.Compose([
    transforms.RandomResizedCrop(TAM, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2),
    transforms.RandomRotation(12),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, DESV),
])

evaluacion = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(TAM),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA, DESV),
])

ds_train = datasets.ImageFolder(DATOS / "train", aumento)
ds_val   = datasets.ImageFolder(DATOS / "val",   evaluacion)
ds_test  = datasets.ImageFolder(DATOS / "test",  evaluacion)

CLASES = ds_train.classes
N_CLASES = len(CLASES)
print("Clases:", CLASES)
print(f"train={len(ds_train)}  val={len(ds_val)}  test={len(ds_test)}")

Clases: ['impresora', 'laptop', 'monitor', 'mouse', 'silla', 'teclado']
train=3292  val=703  test=711


### Desbalance de clases

El dataset no está perfectamente balanceado: `impresora` tiene aproximadamente
la mitad de muestras que `mouse` o `silla`. Open Images simplemente contiene
menos fotografías de impresoras con cajas suficientemente grandes, y ese es el
techo real del dataset.

Se compensa con un muestreador ponderado: durante el entrenamiento, las
muestras de clases escasas se eligen con mayor probabilidad, de modo que cada
lote queda aproximadamente equilibrado sin descartar datos ni duplicar
archivos en disco.

In [4]:
conteos = np.bincount([y for _, y in ds_train.samples], minlength=N_CLASES)
for nombre, n in zip(CLASES, conteos):
    print(f"  {nombre:10s} {n:4d}  {'█' * int(n / 25)}")
print(f"\nProporción máx/mín: {conteos.max() / conteos.min():.2f} a 1")

# Peso inverso a la frecuencia: las clases escasas se muestrean más seguido
peso_clase = 1.0 / conteos
peso_muestra = [peso_clase[y] for _, y in ds_train.samples]
muestreador = WeightedRandomSampler(peso_muestra, len(peso_muestra), replacement=True)

  impresora   339  █████████████
  laptop      593  ███████████████████████
  monitor     625  █████████████████████████
  mouse       630  █████████████████████████
  silla       630  █████████████████████████
  teclado     475  ███████████████████

Proporción máx/mín: 1.86 a 1


In [5]:
LOTE = 32
# num_workers=0 evita problemas de compartición de memoria con MPS
cargador_train = DataLoader(ds_train, batch_size=LOTE, sampler=muestreador, num_workers=0)
cargador_val   = DataLoader(ds_val,   batch_size=LOTE, shuffle=False, num_workers=0)
cargador_test  = DataLoader(ds_test,  batch_size=LOTE, shuffle=False, num_workers=0)
print("lotes por época:", len(cargador_train))

lotes por época: 103


## 3. Modelo

Se parte de MobileNetV3-Large con pesos de ImageNet y se sustituye la última
capa lineal por una de 6 salidas.

El entrenamiento ocurre en dos fases:

1. **Cabeza únicamente.** El extractor de características queda congelado y
   solo se entrena el clasificador nuevo. Evita que los gradientes grandes de
   una capa inicializada al azar destruyan los pesos preentrenados.
2. **Ajuste fino.** Se descongela el último bloque convolucional con una tasa
   de aprendizaje diez veces menor, para adaptar las características de alto
   nivel al dominio específico sin olvidar lo aprendido en ImageNet.

In [6]:
def construir_modelo():
    m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    for p in m.parameters():
        p.requires_grad = False
    entradas = m.classifier[3].in_features
    m.classifier[3] = nn.Linear(entradas, N_CLASES)   # capa nueva, entrenable
    return m.to(dispositivo)

modelo = construir_modelo()
total = sum(p.numel() for p in modelo.parameters())
entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Parámetros totales:    {total:,}")
print(f"Parámetros entrenables: {entrenables:,}  ({entrenables/total:.1%})")

Parámetros totales:    4,209,718
Parámetros entrenables: 7,686  (0.2%)


In [7]:
criterio = nn.CrossEntropyLoss(label_smoothing=0.05)

def pasar_epoca(cargador, optimizador=None):
    """Ejecuta una época. Sin optimizador, corre en modo evaluación."""
    entrenando = optimizador is not None
    modelo.train(entrenando)
    perdida_total, aciertos, vistos = 0.0, 0, 0

    with torch.set_grad_enabled(entrenando):
        for x, y in cargador:
            x, y = x.to(dispositivo), y.to(dispositivo)
            salida = modelo(x)
            perdida = criterio(salida, y)
            if entrenando:
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()
            perdida_total += perdida.item() * y.size(0)
            aciertos += (salida.argmax(1) == y).sum().item()
            vistos += y.size(0)
    return perdida_total / vistos, aciertos / vistos

### Fase 1 — entrenamiento de la cabeza

In [8]:
historial = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "fase": []}
mejor_acc, mejor_estado = 0.0, None

optimizador = torch.optim.AdamW(
    [p for p in modelo.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)

EPOCAS_F1 = 6
for epoca in range(1, EPOCAS_F1 + 1):
    t0 = time.time()
    tl, ta = pasar_epoca(cargador_train, optimizador)
    vl, va = pasar_epoca(cargador_val)
    historial["train_loss"].append(tl); historial["train_acc"].append(ta)
    historial["val_loss"].append(vl);   historial["val_acc"].append(va)
    historial["fase"].append(1)
    if va > mejor_acc:
        mejor_acc = va
        mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
    print(f"época {epoca}/{EPOCAS_F1}  train {tl:.3f}/{ta:.3f}  "
          f"val {vl:.3f}/{va:.3f}  ({time.time()-t0:.0f}s)")

época 1/6  train 1.046/0.753  val 0.673/0.882  (9s)


época 2/6  train 0.676/0.849  val 0.607/0.883  (9s)


época 3/6  train 0.628/0.860  val 0.583/0.885  (9s)


época 4/6  train 0.581/0.886  val 0.578/0.876  (9s)


época 5/6  train 0.553/0.890  val 0.574/0.876  (9s)


época 6/6  train 0.532/0.898  val 0.570/0.881  (9s)


### Fase 2 — ajuste fino del último bloque

In [9]:
# Se descongela solo la parte final del extractor: las capas iniciales
# detectan bordes y texturas genéricos que no conviene alterar.
for capa in modelo.features[-3:]:
    for p in capa.parameters():
        p.requires_grad = True

optimizador = torch.optim.AdamW([
    {"params": [p for p in modelo.features[-3:].parameters()], "lr": 1e-4},
    {"params": modelo.classifier.parameters(), "lr": 3e-4},
], weight_decay=1e-4)

EPOCAS_F2 = 8
planificador = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=EPOCAS_F2)

for epoca in range(1, EPOCAS_F2 + 1):
    t0 = time.time()
    tl, ta = pasar_epoca(cargador_train, optimizador)
    vl, va = pasar_epoca(cargador_val)
    planificador.step()
    historial["train_loss"].append(tl); historial["train_acc"].append(ta)
    historial["val_loss"].append(vl);   historial["val_acc"].append(va)
    historial["fase"].append(2)
    if va > mejor_acc:
        mejor_acc = va
        mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
    print(f"época {epoca}/{EPOCAS_F2}  train {tl:.3f}/{ta:.3f}  "
          f"val {vl:.3f}/{va:.3f}  ({time.time()-t0:.0f}s)")

print(f"\nMejor exactitud en validación: {mejor_acc:.4f}")
modelo.load_state_dict(mejor_estado)   # se recupera el mejor punto, no el último

época 1/8  train 0.486/0.914  val 0.497/0.909  (10s)


época 2/8  train 0.441/0.930  val 0.484/0.917  (10s)


época 3/8  train 0.415/0.944  val 0.471/0.917  (10s)


época 4/8  train 0.394/0.950  val 0.469/0.915  (10s)


época 5/8  train 0.383/0.961  val 0.462/0.929  (10s)


época 6/8  train 0.371/0.966  val 0.459/0.927  (10s)


época 7/8  train 0.362/0.968  val 0.455/0.929  (10s)


época 8/8  train 0.371/0.964  val 0.454/0.930  (11s)

Mejor exactitud en validación: 0.9303


<All keys matched successfully>

## 4. Curvas de entrenamiento

La separación entre las curvas de entrenamiento y validación indica
sobreajuste. Con aumento de datos y un extractor congelado en la primera fase,
se espera que ambas evolucionen juntas.

In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
epocas = range(1, len(historial["train_loss"]) + 1)
corte = historial["fase"].index(2) + 0.5

for eje, clave, titulo in ((ejes[0], "loss", "Pérdida"), (ejes[1], "acc", "Exactitud")):
    eje.plot(epocas, historial[f"train_{clave}"], label="entrenamiento", marker="o", ms=3)
    eje.plot(epocas, historial[f"val_{clave}"], label="validación", marker="s", ms=3)
    eje.axvline(corte, color="gray", ls="--", lw=1)
    eje.text(corte + 0.1, eje.get_ylim()[1]*0.95, "ajuste fino", fontsize=8, color="gray")
    eje.set_xlabel("época"); eje.set_title(titulo); eje.legend(); eje.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(SALIDA / "curvas_entrenamiento.png", dpi=130, bbox_inches="tight")
plt.show()

/var/folders/gs/x1pcvl_97l90xx1fk7c3j4l00000gn/T/ipykernel_12579/2062123862.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Evaluación sobre el conjunto de prueba

El conjunto de prueba no se usó en ningún momento del entrenamiento ni para
seleccionar el mejor punto de control. Es la única estimación honesta del
rendimiento sobre datos nuevos.

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

modelo.eval()
y_real, y_pred, confianzas = [], [], []
with torch.no_grad():
    for x, y in cargador_test:
        p = torch.softmax(modelo(x.to(dispositivo)), dim=1).cpu()
        y_real.extend(y.tolist())
        y_pred.extend(p.argmax(1).tolist())
        confianzas.extend(p.max(1).values.tolist())

exactitud = accuracy_score(y_real, y_pred)
print(f"Exactitud global en prueba: {exactitud:.4f}\n")
print(classification_report(y_real, y_pred, target_names=CLASES, digits=3))

Exactitud global en prueba: 0.9241

              precision    recall  f1-score   support

   impresora      0.986     0.907     0.944        75
      laptop      0.945     0.812     0.874       128
     monitor      0.889     0.948     0.918       135
       mouse      0.985     0.985     0.985       135
       silla      0.961     0.911     0.935       135
     teclado      0.808     0.981     0.886       103

    accuracy                          0.924       711
   macro avg      0.929     0.924     0.924       711
weighted avg      0.930     0.924     0.924       711



In [12]:
import seaborn as sns

mc = confusion_matrix(y_real, y_pred)
mc_norm = mc.astype(float) / mc.sum(axis=1, keepdims=True)

fig, ejes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(mc, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=CLASES, yticklabels=CLASES, ax=ejes[0])
ejes[0].set_title("Matriz de confusión (conteos)")
sns.heatmap(mc_norm, annot=True, fmt=".2f", cmap="Blues", cbar=False, vmin=0, vmax=1,
            xticklabels=CLASES, yticklabels=CLASES, ax=ejes[1])
ejes[1].set_title("Normalizada por clase real")
for eje in ejes:
    eje.set_xlabel("predicción"); eje.set_ylabel("real")

plt.tight_layout()
plt.savefig(SALIDA / "matriz_confusion.png", dpi=130, bbox_inches="tight")
plt.show()

/var/folders/gs/x1pcvl_97l90xx1fk7c3j4l00000gn/T/ipykernel_12579/1599280839.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Análisis de las confusiones

Interesa saber **dónde** falla el modelo, no solo cuánto. Las confusiones entre
clases visualmente parecidas son esperables; las que ocurren entre categorías
muy distintas suelen apuntar a ruido en las etiquetas del dataset.

In [13]:
pares = []
for i in range(N_CLASES):
    for j in range(N_CLASES):
        if i != j and mc[i, j] > 0:
            pares.append((mc[i, j], mc_norm[i, j], CLASES[i], CLASES[j]))
pares.sort(reverse=True)

print("Confusiones más frecuentes:")
for n, tasa, real, pred in pares[:6]:
    print(f"  {real:10s} -> {pred:10s}  {n:3d} casos  ({tasa:.1%} de los {real})")

conf = np.array(confianzas); correcto = np.array(y_real) == np.array(y_pred)
print(f"\nConfianza media en aciertos: {conf[correcto].mean():.3f}")
print(f"Confianza media en errores:  {conf[~correcto].mean():.3f}")

Confusiones más frecuentes:
  laptop     -> teclado      12 casos  (9.4% de los laptop)
  laptop     -> monitor       9 casos  (7.0% de los laptop)
  silla      -> teclado       6 casos  (4.4% de los silla)
  silla      -> monitor       4 casos  (3.0% de los silla)
  monitor    -> teclado       4 casos  (3.0% de los monitor)
  impresora  -> monitor       3 casos  (4.0% de los impresora)

Confianza media en aciertos: 0.884
Confianza media en errores:  0.595


Que la confianza media sea menor en los errores que en los aciertos es una
propiedad deseable: permite al servicio marcar como *incierta* una predicción
por debajo de cierto umbral, en lugar de afirmar algo equivocado con
seguridad.

## 6. Exportación a ONNX

El servicio de inferencia no usa PyTorch sino **onnxruntime**, que pesa
alrededor de 100 MB frente a los ~800 MB del paquete completo de PyTorch. Esto
reduce el tamaño de la imagen Docker, acelera el arranque en frío del
contenedor y baja el consumo de memoria en Fargate.

El eje del lote se declara dinámico para permitir predicciones por lotes en el
futuro sin volver a exportar.

In [14]:
ruta_onnx = SALIDA / "clasificador.onnx"
modelo_cpu = modelo.to("cpu").eval()
ejemplo = torch.randn(1, 3, TAM, TAM)

torch.onnx.export(
    modelo_cpu, ejemplo, ruta_onnx,
    input_names=["imagen"], output_names=["logits"],
    dynamic_axes={"imagen": {0: "lote"}, "logits": {0: "lote"}},
    opset_version=17,
    dynamo=False,   # exportador clásico: grafo estable y reproducible
)
print(f"Exportado: {ruta_onnx.name}  ({ruta_onnx.stat().st_size/1e6:.1f} MB)")

/var/folders/gs/x1pcvl_97l90xx1fk7c3j4l00000gn/T/ipykernel_12579/692413131.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exportado: clasificador.onnx  (16.8 MB)


### Verificación de paridad

Exportar puede introducir diferencias numéricas. Se comprueba que las salidas
de ONNX coincidan con las de PyTorch sobre imágenes reales de prueba: si
divergieran, el modelo desplegado no sería el evaluado.

In [15]:
import onnxruntime as ort

sesion = ort.InferenceSession(str(ruta_onnx), providers=["CPUExecutionProvider"])

lote, _ = next(iter(cargador_test))
with torch.no_grad():
    salida_torch = modelo_cpu(lote).numpy()
salida_onnx = sesion.run(None, {"imagen": lote.numpy()})[0]

dif = np.abs(salida_torch - salida_onnx).max()
iguales = (salida_torch.argmax(1) == salida_onnx.argmax(1)).mean()
print(f"Diferencia numérica máxima: {dif:.2e}")
print(f"Predicciones coincidentes:  {iguales:.1%}")
assert iguales == 1.0, "ONNX y PyTorch discrepan en alguna predicción"
print("\nParidad verificada")

Diferencia numérica máxima: 1.48e-05
Predicciones coincidentes:  100.0%

Paridad verificada


## 7. Latencia de inferencia

El tiempo de respuesta es criterio de evaluación. Se mide sobre CPU, que es
donde correrá el modelo en producción (Fargate no tiene GPU).

In [16]:
entrada = np.random.randn(1, 3, TAM, TAM).astype(np.float32)
for _ in range(10):                      # calentamiento
    sesion.run(None, {"imagen": entrada})

tiempos = []
for _ in range(100):
    t0 = time.perf_counter()
    sesion.run(None, {"imagen": entrada})
    tiempos.append((time.perf_counter() - t0) * 1000)

tiempos = np.array(tiempos)
print(f"Latencia por imagen (CPU, lote=1)")
print(f"  mediana : {np.median(tiempos):.1f} ms")
print(f"  media   : {tiempos.mean():.1f} ms")
print(f"  p95     : {np.percentile(tiempos, 95):.1f} ms")

Latencia por imagen (CPU, lote=1)
  mediana : 2.2 ms
  media   : 2.2 ms
  p95     : 2.2 ms


## 8. Metadatos del modelo

El servicio necesita saber el orden de las clases y los parámetros de
normalización. Guardarlos junto al modelo evita que una discrepancia entre
entrenamiento e inferencia pase inadvertida: si el servicio normalizara con
otros valores, las predicciones serían silenciosamente incorrectas.

In [17]:
reporte = classification_report(y_real, y_pred, target_names=CLASES,
                                output_dict=True, digits=4)

metadatos = {
    "arquitectura": "mobilenet_v3_large",
    "clases": CLASES,
    "tam_entrada": TAM,
    "normalizacion": {"media": MEDIA, "desv": DESV},
    "metricas": {
        "exactitud_prueba": round(exactitud, 4),
        "exactitud_validacion": round(mejor_acc, 4),
        "f1_macro": round(reporte["macro avg"]["f1-score"], 4),
        "por_clase": {
            c: {"precision": round(reporte[c]["precision"], 4),
                "recall": round(reporte[c]["recall"], 4),
                "f1": round(reporte[c]["f1-score"], 4),
                "soporte": int(reporte[c]["support"])}
            for c in CLASES
        },
    },
    "latencia_cpu_ms": {
        "mediana": round(float(np.median(tiempos)), 2),
        "p95": round(float(np.percentile(tiempos, 95)), 2),
    },
    "dataset": {
        "fuente": "Open Images V7 (recortes por caja delimitadora)",
        "train": len(ds_train), "val": len(ds_val), "test": len(ds_test),
    },
}

(SALIDA / "metadatos.json").write_text(json.dumps(metadatos, indent=2, ensure_ascii=False))
print(json.dumps(metadatos["metricas"], indent=2, ensure_ascii=False))

{
  "exactitud_prueba": 0.9241,
  "exactitud_validacion": 0.9303,
  "f1_macro": 0.9237,
  "por_clase": {
    "impresora": {
      "precision": 0.9855,
      "recall": 0.9067,
      "f1": 0.9444,
      "soporte": 75
    },
    "laptop": {
      "precision": 0.9455,
      "recall": 0.8125,
      "f1": 0.8739,
      "soporte": 128
    },
    "monitor": {
      "precision": 0.8889,
      "recall": 0.9481,
      "f1": 0.9176,
      "soporte": 135
    },
    "mouse": {
      "precision": 0.9852,
      "recall": 0.9852,
      "f1": 0.9852,
      "soporte": 135
    },
    "silla": {
      "precision": 0.9609,
      "recall": 0.9111,
      "f1": 0.9354,
      "soporte": 135
    },
    "teclado": {
      "precision": 0.808,
      "recall": 0.9806,
      "f1": 0.886,
      "soporte": 103
    }
  }
}


## Resumen

Artefactos generados en `inventory-ml/modelo/`:

| Archivo | Contenido |
|---|---|
| `clasificador.onnx` | Modelo listo para el servicio de inferencia |
| `metadatos.json` | Clases, normalización y métricas de evaluación |
| `curvas_entrenamiento.png` | Evolución de pérdida y exactitud |
| `matriz_confusion.png` | Desempeño por clase sobre el conjunto de prueba |

El siguiente paso es el servicio FastAPI que carga el modelo ONNX y expone el
endpoint de predicción.